# Imports

In [15]:
# Importing the libraries used in this notebook
import time
import random
import pandas as pd                                  # small result tables (sizes, times) are handled in pandas, not Spark
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel
import os 
print("Packages Versions ->  PySpark:", pyspark.__version__, "| pandas:", pd.__version__)

Packages Versions ->  PySpark: 4.2.0 | pandas: 2.3.3


# Formatos de armazenamento e codecs de compressão

Este notebook compara o formato legado orientado a linha (CSV) com o formato colunar (Parquet), com e sem compressão, usando a base limpa L0 do notebook 00. Para cada combinação de formato e codec são medidos o espaço ocupado no HDFS, o tempo de escrita e o tempo de leitura no Spark. É a linha de base de armazenamento do projeto.

In [4]:
# Initializing the Spark session
N_PART = 8   # same number of output files for every format, so file count is not a hidden variable
N_REPS = 5   # timed repetitions per read, after one discarded warm-up round (Hoefler and Belli, 2015)

spark = (SparkSession.builder
         .master("local[4]")                                   # local mode using the 4 cores of the VM
         .appName("ca1-01-formats")
         .config("spark.driver.memory", "3g")                  # leaves room for the HDFS daemons and the OS in 7.3 GB
         .config("spark.sql.session.timeZone", "UTC")          # avoids the Dublin DST shift on 31/03 with timestamp_ntz
         .config("spark.sql.shuffle.partitions", N_PART)       # makes the orderBy below produce N_PART partitions
         .getOrCreate())
spark

26/10/01 11:04:36 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [5]:
# Defining the paths and the HDFS size helper
BASE = "hdfs://localhost:9000/ca1"
CLEAN = f"{BASE}/clean/yellow_2024q1"     # clean L0 base written in notebook 00
OUT = f"{BASE}/formats"                   # one sub-folder per format/codec combination
MODEL_COLS = ["tpep_pickup_datetime", "trip_distance", "PULocationID", "DOLocationID", "duration_s_true"]   # only the columns the MLP needs

hadoop = spark._jvm.org.apache.hadoop     # Hadoop Java API, reached through Spark's JVM gateway
fs = hadoop.fs.FileSystem.get(spark._jvm.java.net.URI(BASE), spark._jsc.hadoopConfiguration())

def hdfs_size(path):
    # total bytes of all files under an HDFS path (same value as the first column of `hdfs dfs -du -s`)
    return fs.getContentSummary(hadoop.fs.Path(path)).getLength()

In [6]:
# Reading the clean base and sorting it by pickup time
base = spark.read.parquet(CLEAN).orderBy("tpep_pickup_datetime")   # time order is the natural TLC order and makes row order deterministic
base.persist(StorageLevel.MEMORY_AND_DISK)                           # every format is written from exactly the same rows; spills to disk if RAM runs out
base.count()                                                         # count() is the action that fills the cache (expected 9,213,529)

9213529

In [7]:
# Checking the partitions and the size of the clean base
print("partitions:", base.rdd.getNumPartitions())                    # should be N_PART
print("clean base on HDFS (MB):", round(hdfs_size(CLEAN) / 1e6, 1))  # reference size of the L0 base (snappy Parquet)

partitions: 8
clean base on HDFS (MB): 202.9


In [8]:
# Writing an untimed warm-up file: the first Spark job pays JVM start-up costs unrelated to the format
base.write.mode("overwrite").parquet(f"{OUT}/_warmup")
fs.delete(hadoop.fs.Path(f"{OUT}/_warmup"), True)   # removed so it does not use HDFS space

True

In [9]:
# Giving name to each format in a random order and measuring size and write time
FORMATS = [("csv", "none"), ("csv", "gzip"),("parquet", "none"), ("parquet", "snappy"), ("parquet", "gzip"), ("parquet", "zstd")]

order = FORMATS.copy() # create a cope os the formarts 
random.Random(1234).shuffle(order)   # creating a random order so no format is always measured first

rows = []
for fmt, codec in order:
    path = f"{OUT}/{fmt}_{codec}"
    writer = base.write.mode("overwrite").option("compression", codec)   # overwrite lets the cell be run 2 more than 1 time. 
    t0 = time.perf_counter()                                             # monotonic high-resolution clock, right tool for durations
    if fmt == "csv":
        writer.option("header", True).csv(path)                          # header keeps column names, CSV stores every value as text
    else:
        writer.parquet(path)                                             # Parquet stores schema and types inside the file
    t_write = time.perf_counter() - t0                                   # write is a Spark action, so the timer covers the whole job
    rows.append({"format": fmt, "codec": codec, "size_mb": hdfs_size(path) / 1e6, "write_s": t_write})
    print(f"{fmt:8}{codec:8}{rows[-1]['size_mb']:10.1f} MB {t_write:8.1f} s")

sizes = pd.DataFrame(rows)

csv     gzip         195.7 MB     48.3 s


parquet none         265.0 MB     12.2 s


parquet zstd         144.3 MB     14.8 s


parquet gzip         148.0 MB     33.8 s


csv     none        1042.1 MB     48.5 s


parquet snappy       195.5 MB     11.8 s


Aqui ja temos achados importantes para adicionar no relatorio. O csv sem nenhuma compressao tem 1.042 MBs, O parquet com zstd tem aproximadamente 7x menos tamanho ( relatar na literatura motivos pelos quais isso acontece ). Isso confirma a escolha do zstd como codecfix ( explicar no relatorio o por que esse nome e qual o fundamento cientifico por tras ). 

o CSV com gzip (195,7 MB) e o Parquet com snappy (195,5 MB) têm praticamente o mesmo tamanho , ou seja , compressão sozinha não explica a vantagem do formato colunar. O que explica é o layout por coluna, a codificação binária com tipos e a leitura só das colunas pedidas. Procurar na literatura comparacoes entre tipos ( *ARTIGOS*) 

In [10]:
# Defining how each format is read
def read_format(fmt, path):
    if fmt == "csv":
        return spark.read.schema(base.schema).option("header", True).csv(path)   # explicit schema: no extra pass for type inference
    return spark.read.parquet(path)                                             # schema comes from the Parquet file footer

In [11]:
# Measuring the read time in rounds, with a new random order in every round
rng = random.Random(1234)
times = []
for rep in range(N_REPS + 1):
    order = [(fmt, codec, scan) for fmt, codec in FORMATS for scan in ["full", "model_cols"]]
    rng.shuffle(order)                                   # a new random order in every round removes the order bias
    for fmt, codec, scan in order:
        df = read_format(fmt, f"{OUT}/{fmt}_{codec}")    # reading is lazy: nothing is read until the action below
        if scan == "model_cols":
            df = df.select(*MODEL_COLS)                  # projection: Parquet can skip the other columns, CSV reads every full line
        t0 = time.perf_counter()
        df.write.format("noop").mode("overwrite").save() # noop sink forces a real read and decode but writes nothing
        dt = time.perf_counter() - t0
        if rep > 0:                                      # round 0 is a full warm-up round and is discarded
            times.append({"format": fmt, "codec": codec, "scan": scan, "rep": rep, "read_s": dt})
    print("round", rep, "done")

times = pd.DataFrame(times)

round 0 done


round 1 done


round 2 done


round 3 done


round 4 done


[Stage 143:>                                                        (0 + 4) / 4]

round 5 done


In [12]:
# Building the summary table
med = times.pivot_table(index=["format", "codec"], columns="scan", values="read_s", aggfunc="median")   # median resists outliers from background activity in the VM
iqr = times.pivot_table(index=["format", "codec"], columns="scan", values="read_s",
                        aggfunc=lambda s: s.quantile(0.75) - s.quantile(0.25))   # interquartile range, reported with the median

summary = (sizes.set_index(["format", "codec"])
           .join(med.add_prefix("read_median_s_"))
           .join(iqr.add_prefix("read_iqr_s_")))
summary["ratio_vs_csv"] = summary.loc[("csv", "none"), "size_mb"] / summary["size_mb"]   # how many times smaller than the uncompressed CSV
display(summary.round(2))

size_mb  write_s  read_median_s_full  \
format  codec                                          
csv     gzip     195.71    48.26               15.41   
parquet none     264.99    12.23                2.29   
        zstd     144.30    14.84                2.29   
        gzip     148.02    33.82                2.42   
csv     none    1042.07    48.47               13.15   
parquet snappy   195.46    11.82                2.10   

                read_median_s_model_cols  read_iqr_s_full  \
format  codec                                               
csv     gzip                        7.23             6.41   
parquet none                        0.88             0.56   
        zstd                        0.87             0.44   
        gzip                        0.96             1.03   
csv     none                        7.04             2.25   
parquet snappy                      0.78             0.06   

                read_iqr_s_model_cols  ratio_vs_csv  
format  codec                                        
csv     gzip                     1.16          5.32  
parquet none                     0.14          3.93  
        zstd                     0.13          7.22  
        gzip                     0.08          7.04  
csv     none                     0.79          1.00  
parquet snappy                   0.02          5.33

Aqui oa resultados foram salvos em 1 pasta chamada results para serem comparados no futuro. 

In [18]:
# Saving the results to an absolute path, so it works wherever the notebook is saved
RESULTS = os.path.expanduser("~/ca1/results")
os.makedirs(RESULTS, exist_ok=True)                          # creates the folder if it does not exist yet
print("notebook folder:", os.getcwd())                       # shows where the notebook is running from
times.to_csv(f"{RESULTS}/01_read_times.csv", index=False)    # raw read times, kept for the report
summary.to_csv(f"{RESULTS}/01_summary.csv")
base.unpersist();                                             # frees the cached base before the next notebook

notebook folder: /home/hduser/ca1


In [17]:
# Stopping the Spark session
spark.stop() 